# Arabic Manuscript Recognition

#### Mohammed Almazyad  |  201761130
#### Meshal Aldajani  |  201935830

## 1. Data Pipeline

### Import libraries

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # must come before importing keras

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import keras
from keras import ops

### Settings

All the numbers that control the notebook are kept in one place.

* Every line image is resized to a height of **32 px** and padded to a width of **1024 px**.
* The image is cut into vertical frames **8 px** wide, moving **4 px** each time. This gives **255 frames**,
  each flattened to **32 x 8 = 256 numbers**.
* Texts longer than **200 characters** are removed, because the CTC loss needs more frames than characters.
* `TRAIN_SUBSET` can be set (for example to 5000) to run the 3 x 3 comparison faster on part of the training data.
  `None` means all training lines are used.

In [ ]:
DATA_FOLDER = r"C:\Users\mesha\OneDrive - KFUPM\Desktop\MX\261\ICS504\AraMs"
CACHE_FOLDER = os.path.join(os.path.expanduser("~"), "arams_cache")   # local folder, outside OneDrive

# image and frame settings
IMG_HEIGHT = 32
IMG_WIDTH = 1024
WIN = 8
STRIDE = 4
NUM_FRAMES = (IMG_WIDTH - WIN) // STRIDE + 1   # 255
FEATURES = IMG_HEIGHT * WIN                    # 256
MAX_LABEL_LEN = 200

# training settings
BATCH_SIZE = 32
EPOCHS = 25          # maximum epochs per run (early stopping may stop sooner)
PATIENCE = 5         # stop if validation CER does not improve for this many epochs
TRAIN_SUBSET = None  # e.g. 5000 to speed up the 3 x 3 grid; None = use all training lines
CER_SUBSET = 1000    # training lines used to measure the training CER after each epoch
SEED = 42

keras.utils.set_random_seed(SEED)
print("Frames per line:", NUM_FRAMES)
print("Numbers per frame:", FEATURES)

### 1.1 Load the dataset

The AraMS release gives one CSV file per split (`train.csv`, `val.csv`, `test.csv`) with the image file name and its
transcription. We do **not** load all images into memory here; we only keep each image's path and text in a table.
The book name is taken from the file name (the part before `__`).

In [ ]:
def load_split(split):
    """Read <split>.csv and return a DataFrame with image_path, text, book and split columns."""
    df = pd.read_csv(os.path.join(DATA_FOLDER, split + ".csv"))
    df["book"] = df["image_path"].apply(lambda name: name.split("__")[0])
    df["split"] = split
    df["image_path"] = df["image_path"].apply(lambda name: os.path.join(DATA_FOLDER, "images", split, name))
    return df


train_df = load_split("train")
val_df = load_split("val")
test_df = load_split("test")

print("Train lines:", len(train_df))
print("Val lines:  ", len(val_df))
print("Test lines: ", len(test_df))
train_df.head()

In [ ]:
# def needed_frames(text):
#     """Minimum number of frames CTC needs to write this text."""
#     repeats = 0
#     for i in range(1, len(text)):
#         if text[i] == text[i - 1]:
#             repeats += 1
#     return len(text) + repeats


# def clean_data(df, max_len=MAX_LABEL_LEN):
#     before = len(df)
#     df = df.dropna(subset=["text"]).copy()
#     df["text"] = df["text"].astype(str).str.strip()
#     df = df[df["text"].str.len() > 0]
#     df = df[df["text"].str.len() <= max_len]
#     df = df[df["text"].apply(needed_frames) <= NUM_FRAMES]
#     df = df.reset_index(drop=True)
#     print(f"Kept {len(df)} of {before} lines")
#     return df


# train_df = clean_data(train_df)
# val_df = clean_data(val_df)
# test_df = clean_data(test_df)

### Display samples

The dataset has no document classes, so we show **one line from each book** (14 books in total).
Matplotlib does not join Arabic letters correctly, so the reference text is printed below the figure.

In [ ]:
def show_samples(df):
    """Show the first line image of every book in df, and print its text."""
    books = sorted(df["book"].unique())
    fig, axes = plt.subplots(len(books), 1, figsize=(10, 1.1 * len(books)))
    if len(books) == 1:
        axes = [axes]

    for i in range(len(books)):
        row = df[df["book"] == books[i]].iloc[0]
        axes[i].imshow(Image.open(row["image_path"]).convert("RGB"))
        axes[i].set_title(f"{i + 1}. {books[i]} ({row['split']})", fontsize=9, loc="left")
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

    for i in range(len(books)):
        row = df[df["book"] == books[i]].iloc[0]
        print(f"{i + 1}. {books[i]}: {row['text']}")


all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
show_samples(all_df)

### Character distribution

For text recognition, the real output "classes" are the characters. This chart shows the 30 most frequent
characters in the training texts.

In [ ]:
def plot_char_frequency(texts, top=30):
    counts = {}
    for text in texts:
        for c in text:
            if c == " ":
                c = "space"
            counts[c] = counts.get(c, 0) + 1
    # print(sorted(counts.items(), key=lambda p: p[1], reverse=True)[:top])
    pairs = sorted(counts.items(), key=lambda p: p[1], reverse=True)[:top]
    labels = [p[0] for p in pairs]
    values = [p[1] for p in pairs]

    plt.figure(figsize=(12, 5.5))
    plt.bar(labels, values, color="#125f66")
    plt.title(f"Top {top} characters in the training texts")
    plt.ylabel("Count")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
    print("Distinct characters in training texts:", len(counts))


plot_char_frequency(train_df["text"])

### 1.2 Preprocessing

Every image has a different size, so the images cannot be stacked into one array or divided by 255 as a list.
Each image is processed **on its own** and made the same size:

1. **Grayscale**: color does not help to read the text.
2. **Resize** to height 32, keeping the width/height ratio.
3. **Normalize** to 0–1 by dividing by 255, then **invert** so ink = 1 and paper = 0.
4. **Flip** left–right so the first frame matches the first letter of the Arabic text (right-to-left reading).
   Known limitation: digits are written left-to-right, so numbers end up reversed.
5. **Pad** to width 1024 with zeros (empty paper).
6. **Reshape** into 255 frames of 256 numbers each.

In [ ]:
def preprocess_image(path):
    """Steps 1-5: return a (32, 1024) array with values between 0 and 1."""
    # 1. grayscale
    img = Image.open(path).convert("L")
    # print('img arr', np.array(img))
    # print('first row', len(np.array(img)[0]))

    # 2. resize to the fixed height, keeping the ratio
    w, h = img.size
    # print('img size', w, h)
    new_w = int(w * IMG_HEIGHT / h)
    # print('new w', new_w)
    new_w = max(1, min(new_w, IMG_WIDTH))
    # print('max new w', new_w)
    img = img.resize((new_w, IMG_HEIGHT))
    # print('img size', img.size)

    # print('new img arr', np.array(img, dtype=np.float32))
    # 3. normalize to 0..1
    arr = np.array(img, dtype=np.float32) / 255.0
    # print('norm arr', arr)
    
    # 3.1 make ink pixels gets higher values
    arr = 1.0 - arr
    # print('inv arr', arr)

    # 4. flip left-right to match the model reading approach
    arr = np.fliplr(arr)
    # print('flip arr', arr)

    # 5. pad on a fixed-size canvas
    canvas = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.float32)
    # print('canvas', canvas)
    canvas[:, :new_w] = arr
    # print('new canvas', canvas)
    return canvas


def image_to_frames(canvas):
    """Step 6: cut the (32, 1024) canvas into (255, 256) frames."""
    frames = []
    for start in range(0, IMG_WIDTH - WIN + 1, STRIDE):
        window = canvas[:, start:start + WIN]    # a 32 x 8 slice
        frames.append(window.flatten())          # 256 numbers
    return np.array(frames, dtype=np.float32)

Here is one image before and after preprocessing:

In [ ]:
sample_path = train_df["image_path"][0]
original = np.array(Image.open(sample_path).convert("L"))
canvas = preprocess_image(sample_path)
frames = image_to_frames(canvas)

print("Original shape:", original.shape, " values", original.min(), "to", original.max())
print("After steps 1-5:", canvas.shape, " values", canvas.min(), "to", canvas.max())
print("After step 6 (frames):", frames.shape)

fig, axes = plt.subplots(2, 1, figsize=(10, 3))
axes[0].imshow(original, cmap="gray")
axes[0].set_title("Original", loc="left")
axes[1].imshow(canvas, cmap="gray")
axes[1].set_title("Grayscale, resized, normalized, inverted, flipped and padded", loc="left")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Preprocess every image once (cache)

The experiments train 14 models. Opening and resizing 20,000 images in every epoch of every run would be very slow,
so each image is preprocessed **once** and saved to a `.npy` file in a local folder. The canvases are stored as
`uint8` (0–255) to save memory and turned back into 0–1 floats when a batch is made.
The first run takes a few minutes; later runs load the file in seconds.

In [ ]:
def cache_images(df, name):
    """Preprocess all images of df once and save them; load the saved file if it already exists."""
    os.makedirs(CACHE_FOLDER, exist_ok=True)
    cache_file = os.path.join(CACHE_FOLDER, f"{name}_{IMG_HEIGHT}x{IMG_WIDTH}.npy")

    if os.path.exists(cache_file):
        images = np.load(cache_file)
        if len(images) == len(df):
            print(f"Loaded {name}: {images.shape}")
            return images

    images = np.zeros((len(df), IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)
    for i in range(len(df)):
        canvas = preprocess_image(df["image_path"][i])
        images[i] = np.round(canvas * 255).astype(np.uint8)
        if (i + 1) % 5000 == 0:
            print(f"  {name}: {i + 1}/{len(df)} images done")
    np.save(cache_file, images)
    print(f"Saved {name}: {images.shape}")
    return images


train_images = cache_images(train_df, "train")
val_images = cache_images(val_df, "val")
test_images = cache_images(test_df, "test")

### Label encoding

The network cannot read letters, so every character gets a number.

* Numbers start from **1**, because **0 is reserved for the CTC "blank"** symbol.
* Every label is **padded with 0** to length 200, so all labels in a batch have the same length.
* The vocabulary is built from the **training texts only**.

In [ ]:
def build_vocab(texts):
    all_chars = set()
    for text in texts:
        for c in text:
            all_chars.add(c)
    all_chars = sorted(all_chars)

    char2idx = {}
    idx2char = {}
    for i, c in enumerate(all_chars):
        char2idx[c] = i + 1        # 0 is the blank
        idx2char[i + 1] = c
    return char2idx, idx2char


def text_to_label(text, char2idx):
    ids = []
    for c in text:
        if c in char2idx:          # skip characters never seen in training
            ids.append(char2idx[c])
    while len(ids) < MAX_LABEL_LEN:
        ids.append(0)              # padding
    return ids


def label_to_text(ids, idx2char):
    text = ""
    for i in ids:
        i = int(i)
        if i != 0:
            text += idx2char[i]
    return text


char2idx, idx2char = build_vocab(train_df["text"])
NUM_CLASSES = len(char2idx) + 1    # +1 for the blank
print("Vocabulary size:", len(char2idx))
print("Output classes (with blank):", NUM_CLASSES)

example = train_df["text"][4]
encoded = text_to_label(example, char2idx)
print("\nText:   ", example)
print("Encoded:", encoded[:len(example)], "... + zeros")
print("Decoded:", label_to_text(encoded, idx2char))

### Conversion to PyTorch tensors

A PyTorch `Dataset` pairs each image with its text and returns both as tensors:

* `__len__` returns the number of lines.
* `__getitem__` takes cached image number *i*, converts it to 0–1 floats, cuts it into frames and encodes its text.

A `DataLoader` groups the samples into batches of 32 and shuffles the training data every epoch.
Keras accepts these loaders directly in `model.fit()`.

In [ ]:
class LineDataset(Dataset):
    def __init__(self, images, texts, char2idx):
        self.images = images            # uint8 array (N, 32, 1024) from cache_images
        self.texts = list(texts)
        self.char2idx = char2idx

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        canvas = self.images[i].astype(np.float32) / 255.0
        x = torch.tensor(image_to_frames(canvas), dtype=torch.float32)
        y = torch.tensor(text_to_label(self.texts[i], self.char2idx), dtype=torch.long)
        return x, y


def make_loader(images, texts, char2idx, shuffle, batch_size=BATCH_SIZE):
    return DataLoader(LineDataset(images, texts, char2idx), batch_size=batch_size, shuffle=shuffle)


def take_subset(images, texts, n, seed=SEED):
    """Pick n random lines (used for faster experiments and for measuring training CER)."""
    texts = list(texts)
    if n is None or n >= len(texts):
        return images, texts
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(texts), size=n, replace=False))
    return images[idx], [texts[i] for i in idx]


train_loader = make_loader(train_images, train_df["text"], char2idx, shuffle=True)
val_loader = make_loader(val_images, val_df["text"], char2idx, shuffle=False)
test_loader = make_loader(test_images, test_df["text"], char2idx, shuffle=False)

grid_images, grid_texts = take_subset(train_images, train_df["text"], TRAIN_SUBSET)
grid_train_loader = make_loader(grid_images, grid_texts, char2idx, shuffle=True)

cer_images, cer_texts = take_subset(train_images, train_df["text"], CER_SUBSET)
train_eval_loader = make_loader(cer_images, cer_texts, char2idx, shuffle=False)

xb, yb = next(iter(train_loader))
print("X batch:", tuple(xb.shape), xb.dtype)    # (32, 255, 256)
print("y batch:", tuple(yb.shape), yb.dtype)    # (32, 200)
print("Lines used for the 3 x 3 grid:", len(grid_texts))

### 1.3 – 1.5 Train / validation / test split

AraMS-28k comes with **published splits** (train / val / test), so we use them as stated in the proposal.
These splits keep each **book in only one split**. A random 80/20 split would put lines of the same book (same
handwriting) in both training and test, which would make the test score look better than it really is.

In [ ]:
train_books = set(train_df["book"])
val_books = set(val_df["book"])
test_books = set(test_df["book"])

print("Train books:", sorted(train_books))
print("Val books:  ", sorted(val_books))
print("Test books: ", sorted(test_books))
print("\nBooks shared between train and val: ", train_books & val_books)
print("Books shared between train and test:", train_books & test_books)
print("Books shared between val and test:  ", val_books & test_books)

total = len(train_df) + len(val_df) + len(test_df)
print(f"\nTrain {len(train_df) / total:.1%} | Val {len(val_df) / total:.1%} | Test {len(test_df) / total:.1%}")

For reuse in other assignments (where no predefined split exists), we also provide a split function:
`split_data(df, 0.2)` gives the 80/20 train/test split, and calling it again on the training part gives the
80/20 train/validation split.

There are no classes to stratify on, so we stratify by **text-length group** (0–29, 30–59, 60–89, ... characters).
This keeps short and long lines in the same proportions in every part. It is demonstrated below; the published
splits above are the ones used for training.

In [ ]:
# Step 1: put every line into a length group
def length_group(text):
    if len(text) < 40:
        return "short"
    elif len(text) < 80:
        return "medium"
    else:
        return "long"


# Step 2: split a table into 80% and 20%, keeping the same mix of short/medium/long lines
def split_data(df, test_size=0.2):
    groups = df["text"].apply(length_group)
    part_80, part_20 = train_test_split(df, test_size=test_size, stratify=groups, random_state=SEED)
    return part_80, part_20


# Step 3: demo - split all lines into train/test, then split train into train/val
demo_trainval, demo_test = split_data(all_df)          # 80% / 20%
demo_train, demo_val = split_data(demo_trainval)       # 80% / 20% of the first part (the 80%)

print("Train:", len(demo_train), " Val:", len(demo_val), " Test:", len(demo_test))

# Step 4: check that every part has the same mix of short/medium/long lines
for name, part in [("all", all_df), ("train", demo_train), ("val", demo_val), ("test", demo_test)]:
    shares = part["text"].apply(length_group).value_counts(normalize=True).round(3)
    print(name, dict(shares))

## 2. MLP Implementation

One function builds any MLP. It receives a **list of hidden-layer sizes**:
`[512, 256]` gives the shallow network (2 hidden layers) and `[512, 512, 256, 256, 128]` gives the deeper network
(5 hidden layers).

Keras applies a `Dense` layer to each of the 255 frames separately, with the **same weights for every frame**,
so the MLP answers the same question for every slice of the line: *which character is here?*

**Choices and justification**

| Choice | Value | Why |
|---|---|---|
| Input | 256 numbers per frame | one 32 x 8 slice of the line |
| Hidden layer 1 | 512 neurons | wider than the input, to learn many different stroke shapes |
| Hidden layer 2 | 256 neurons | narrows the features before the output, keeping the model small |
| Activation | ReLU | fast, simple, and avoids the vanishing gradients of sigmoid/tanh |
| Regularization | Dropout 0.2 and L2 1e-4 after every hidden layer | reduces overfitting to the 9 training books |
| Output layer | `Dense(NUM_CLASSES)`, no activation | one score per character + blank for every frame; the CTC loss applies softmax itself |

The deeper network keeps the same first layer and adds layers that slowly narrow (512 → 512 → 256 → 256 → 128),
so the only difference between the two networks is depth.

In [ ]:
L2_STRENGTH = 1e-4
DROPOUT = 0.2

SHALLOW_UNITS = [512, 256]
DEEP_UNITS = [512, 512, 256, 256, 128]


def build_mlp(hidden_units, num_classes=NUM_CLASSES, activation="relu",
              dropout=DROPOUT, l2=L2_STRENGTH, name="mlp"):
    """Frame-wise MLP: (batch, 255, 256) -> (batch, 255, num_classes) character scores."""
    inputs = keras.Input(shape=(NUM_FRAMES, FEATURES))
    x = inputs
    for units in hidden_units:
        x = keras.layers.Dense(units, activation=activation,
                               kernel_regularizer=keras.regularizers.L2(l2))(x)
        x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(num_classes, name="char_scores")(x)
    return keras.Model(inputs, outputs, name=name)


build_mlp(SHALLOW_UNITS, name="shallow_mlp").summary()
build_mlp(DEEP_UNITS, name="deep_mlp").summary()

## 3. Required Experiments

### 3.1 Loss functions

The task is sequence prediction, so the three classification losses from the assignment are used in their
**sequence (CTC) form**:

| Assignment option | Our version | What it does |
|---|---|---|
| Cross-entropy | **CTC** | the standard loss for line recognition; it sums the probability of every way the 255 frames can spell the target text |
| Cross-entropy + label smoothing | **CTC + entropy penalty** | subtracts β × (entropy of the frame predictions), which stops the model from becoming over-confident; this is the same idea as label smoothing |
| Focal loss | **Focal CTC** | multiplies the loss by (1 − p)^γ, where p is the model's confidence per character, so easy lines count less and hard lines count more |

In focal CTC, p is computed per character, p = exp(−CTC / text length), so long lines are not treated as hard just
because they are long. The weight is treated as a constant during backpropagation (`stop_gradient`).

**Regularization strength (reported)**

| Setting | Value |
|---|---|
| L2 weight decay (every Dense layer) | 1e-4 |
| Dropout (after every hidden layer) | 0.2 |
| Entropy penalty β (CTC + entropy) | 0.02 |
| Focal γ (focal CTC) | 2.0 |
| Gradient clipping (all optimizers) | global norm 5.0 |

The three losses have different scales, so loss values are **not** compared between them. Runs are compared by
**validation CER**.

In [ ]:
ENTROPY_BETA = 0.02
FOCAL_GAMMA = 2.0


def ctc_per_line(y_true, y_pred):
    """Return the CTC loss of every line in the batch, and the true text lengths."""
    y_true = ops.cast(y_true, "int32")                    # Keras may pass labels as floats
    label_len = ops.sum(ops.cast(ops.not_equal(y_true, 0), "int32"), axis=1)
    batch_size = ops.shape(y_pred)[0]
    frame_len = ops.full((batch_size,), NUM_FRAMES, dtype="int32")
    loss = ops.ctc_loss(y_true, y_pred, label_len, frame_len, mask_index=0)
    return loss, label_len


def ctc_loss(y_true, y_pred):
    loss, _ = ctc_per_line(y_true, y_pred)
    return loss


def make_ctc_entropy_loss(beta=ENTROPY_BETA):
    def ctc_entropy_loss(y_true, y_pred):
        loss, _ = ctc_per_line(y_true, y_pred)
        probs = ops.softmax(y_pred, axis=-1)
        log_probs = ops.log_softmax(y_pred, axis=-1)
        entropy = -ops.sum(probs * log_probs, axis=-1)    # entropy of every frame
        entropy = ops.sum(entropy, axis=1)                # total for the line
        return loss - beta * entropy
    return ctc_entropy_loss


def make_focal_ctc_loss(gamma=FOCAL_GAMMA):
    def focal_ctc_loss(y_true, y_pred):
        loss, label_len = ctc_per_line(y_true, y_pred)
        per_char = loss / ops.cast(label_len, "float32")
        p = ops.exp(-per_char)                            # confidence per character (0..1)
        weight = ops.stop_gradient(ops.power(1.0 - p, gamma))
        return weight * loss
    return focal_ctc_loss


LOSSES = {
    "CTC": ctc_loss,
    "CTC + entropy": make_ctc_entropy_loss(),
    "Focal CTC": make_focal_ctc_loss(),
}
print("Loss functions:", list(LOSSES))

### 3.2 Optimizers

The three required optimizers. In the 3 x 3 comparison, each optimizer starts from its usual learning rate
(SGD 0.01, RMSprop 0.001, Adam 0.001); the learning rate is tuned afterwards for the best combination.
SGD uses momentum 0.9. All optimizers clip the gradient norm to 5, because CTC can produce large gradients early in
training.

In [ ]:
CLIP_NORM = 5.0
DEFAULT_LR = {"SGD + momentum": 0.01, "RMSprop": 0.001, "Adam": 0.001}


def get_optimizer(name, lr):
    if name == "SGD + momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9, clipnorm=CLIP_NORM)
    if name == "RMSprop":
        return keras.optimizers.RMSprop(learning_rate=lr, clipnorm=CLIP_NORM)
    if name == "Adam":
        return keras.optimizers.Adam(learning_rate=lr, clipnorm=CLIP_NORM)
    raise ValueError("Unknown optimizer: " + name)


OPTIMIZERS = list(DEFAULT_LR)
print("Optimizers:", OPTIMIZERS)

### 3.3 Performance metric: Character Error Rate (CER)

1. **Greedy decoding**: for every frame take the most likely character, then remove repeated characters and blanks.
2. **Edit distance**: the number of insertions, deletions and substitutions needed to turn the prediction into the
   true text.
3. **CER** = total edit distance / total number of true characters. **WER** is the same at word level and is reported
   on the test set as extra information.

A callback measures the training CER (on 1,000 training lines) and the validation CER after every epoch. It also
keeps the weights of the epoch with the **lowest validation CER** and stops training early if the validation CER does
not improve for `PATIENCE` epochs.

In [ ]:
def greedy_decode(frame_scores, idx2char):
    best = np.argmax(frame_scores, axis=-1)   # best class for each frame
    text = ""
    prev = 0
    for idx in best:
        idx = int(idx)
        if idx != prev and idx != 0:          # skip repeats and blanks
            text += idx2char[idx]
        prev = idx
    return text


def edit_distance(a, b):
    """Number of insertions, deletions and substitutions to turn sequence a into sequence b."""
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[-1]


def predict_texts(model, loader, idx2char=idx2char):
    """Return two lists: predicted texts and true texts for every line in the loader."""
    preds, trues = [], []
    for xb, yb in loader:
        with torch.no_grad():
            scores = ops.convert_to_numpy(model(xb, training=False))
        labels = np.asarray(yb)
        for i in range(len(labels)):
            preds.append(greedy_decode(scores[i], idx2char))
            trues.append(label_to_text(labels[i], idx2char))
    return preds, trues


def error_rates(preds, trues):
    char_errors = sum(edit_distance(p, t) for p, t in zip(preds, trues))
    word_errors = sum(edit_distance(p.split(), t.split()) for p, t in zip(preds, trues))
    cer = char_errors / sum(len(t) for t in trues)
    wer = word_errors / sum(len(t.split()) for t in trues)
    return cer, wer


def compute_cer(model, loader):
    preds, trues = predict_texts(model, loader)
    return error_rates(preds, trues)[0]


class CERTracker(keras.callbacks.Callback):
    """After every epoch: record loss and CER, keep the best weights, and stop early."""

    def __init__(self, train_eval_loader, val_loader, patience=PATIENCE):
        super().__init__()
        self.train_eval_loader = train_eval_loader
        self.val_loader = val_loader
        self.patience = patience
        self.history = {"loss": [], "val_loss": [], "cer": [], "val_cer": []}
        self.best_val_cer = float("inf")
        self.best_epoch = 0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        train_cer = compute_cer(self.model, self.train_eval_loader)
        val_cer = compute_cer(self.model, self.val_loader)
        self.history["loss"].append(float(logs["loss"]))
        self.history["val_loss"].append(float(logs["val_loss"]))
        self.history["cer"].append(train_cer)
        self.history["val_cer"].append(val_cer)
        print(f"  epoch {epoch + 1:2d}: loss {logs['loss']:.2f}  val_loss {logs['val_loss']:.2f}  "
              f"train CER {train_cer:.3f}  val CER {val_cer:.3f}")

        if val_cer < self.best_val_cer:
            self.best_val_cer = val_cer
            self.best_epoch = epoch + 1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print("  early stopping")
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

### Experiment runner

One function trains one configuration and returns the trained model (with its best weights), its history, and a
row for the results table. All experiments below call this function.

In [ ]:
def run_experiment(hidden_units, loss_name, optimizer_name, lr, train_loader, epochs=EPOCHS, name="mlp"):
    keras.utils.set_random_seed(SEED)                 # same starting weights for a fair comparison
    model = build_mlp(hidden_units, name=name)
    model.compile(optimizer=get_optimizer(optimizer_name, lr), loss=LOSSES[loss_name])
    tracker = CERTracker(train_eval_loader, val_loader)

    start = time.time()
    model.fit(train_loader, validation_data=val_loader, epochs=epochs, callbacks=[tracker], verbose=0)
    minutes = (time.time() - start) / 60

    result = {
        "loss": loss_name,
        "optimizer": optimizer_name,
        "learning rate": lr,
        "best val CER": round(tracker.best_val_cer, 4),
        "best epoch": tracker.best_epoch,
        "epochs run": len(tracker.history["val_cer"]),
        "time (min)": round(minutes, 1),
    }
    return model, tracker.history, result

### 3.4 Comparison 1: 3 x 3 grid of loss functions and optimizers (shallow MLP)

Every combination of the three losses and three optimizers is trained with the shallow MLP. The results are saved to
a CSV file after each run, so nothing is lost if the notebook stops.

In [ ]:
grid_results = []
for loss_name in LOSSES:
    for optimizer_name in OPTIMIZERS:
        lr = DEFAULT_LR[optimizer_name]
        print(f"\n=== {loss_name} | {optimizer_name} | lr = {lr} ===")
        _, _, result = run_experiment(SHALLOW_UNITS, loss_name, optimizer_name, lr,
                                      grid_train_loader, name="shallow_mlp")
        grid_results.append(result)
        pd.DataFrame(grid_results).to_csv("grid_results.csv", index=False)

grid_table = pd.DataFrame(grid_results)
grid_table

Validation CER of each combination (rows = loss function, columns = optimizer). Lower is better.

In [ ]:
grid_pivot = grid_table.pivot(index="loss", columns="optimizer", values="best val CER")
grid_pivot = grid_pivot.loc[list(LOSSES), OPTIMIZERS]
display(grid_pivot.style.highlight_min(axis=None, color="#bfe3df").format("{:.4f}"))

best_row = grid_table.sort_values("best val CER").iloc[0]
BEST_LOSS = best_row["loss"]
BEST_OPTIMIZER = best_row["optimizer"]
print(f"Best combination: {BEST_LOSS} + {BEST_OPTIMIZER}  (validation CER {best_row['best val CER']:.4f})")

### 3.5 Comparison 2: four learning rates for the best combination

The best loss–optimizer pair is trained with four learning rates around the optimizer's usual value, on the full
training set.

In [ ]:
LR_CANDIDATES = {
    "SGD + momentum": [0.1, 0.03, 0.01, 0.003],
    "RMSprop": [0.003, 0.001, 0.0003, 0.0001],
    "Adam": [0.003, 0.001, 0.0003, 0.0001],
}

lr_results = []
lr_models = {}
lr_histories = {}
for lr in LR_CANDIDATES[BEST_OPTIMIZER]:
    print(f"\n=== {BEST_LOSS} | {BEST_OPTIMIZER} | lr = {lr} ===")
    model, history, result = run_experiment(SHALLOW_UNITS, BEST_LOSS, BEST_OPTIMIZER, lr,
                                            train_loader, name="shallow_mlp")
    lr_results.append(result)
    lr_models[lr] = model
    lr_histories[lr] = history
    pd.DataFrame(lr_results).to_csv("lr_results.csv", index=False)

lr_table = pd.DataFrame(lr_results)
display(lr_table.style.highlight_min(subset=["best val CER"], color="#bfe3df"))

BEST_LR = lr_table.sort_values("best val CER").iloc[0]["learning rate"]
print(f"Best learning rate: {BEST_LR}")

shallow_model = lr_models[BEST_LR]          # shallow MLP with the optimal settings
shallow_history = lr_histories[BEST_LR]

### 3.6 Deeper MLP (5 hidden layers) with the optimal settings

The deeper network uses the best loss function, optimizer and learning rate found above.

In [ ]:
print(f"=== Deep MLP | {BEST_LOSS} | {BEST_OPTIMIZER} | lr = {BEST_LR} ===")
deep_model, deep_history, deep_result = run_experiment(DEEP_UNITS, BEST_LOSS, BEST_OPTIMIZER, BEST_LR,
                                                       train_loader, name="deep_mlp")
pd.DataFrame([deep_result])

### 3.7 Comparison 3: shallow vs deeper network on the test set

The test set (3 books never seen during training or model selection) is used only here.

In [ ]:
def evaluate_on_test(model, name, hidden_units):
    preds, trues = predict_texts(model, test_loader)
    cer, wer = error_rates(preds, trues)
    test_loss = model.evaluate(test_loader, verbose=0)
    row = {
        "model": name,
        "hidden layers": len(hidden_units),
        "hidden units": str(hidden_units),
        "parameters": model.count_params(),
        "test loss": round(float(test_loss), 3),
        "test CER": round(cer, 4),
        "test WER": round(wer, 4),
    }
    return row, preds, trues


shallow_row, shallow_preds, test_trues = evaluate_on_test(shallow_model, "Shallow MLP", SHALLOW_UNITS)
deep_row, deep_preds, _ = evaluate_on_test(deep_model, "Deep MLP", DEEP_UNITS)

test_table = pd.DataFrame([shallow_row, deep_row])
print(f"Settings: {BEST_LOSS} | {BEST_OPTIMIZER} | lr = {BEST_LR}")
test_table

A few test lines with the predictions of both networks:

In [ ]:
for i in range(5):
    print("True:   ", test_trues[i])
    print("Shallow:", shallow_preds[i])
    print("Deep:   ", deep_preds[i])
    print()

## 4. Required Plots

Training and validation **loss** and **CER** per epoch for both networks with the optimal settings.
The training loss includes dropout and the L2 penalty, so it can be higher than the validation loss.

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["loss"], marker="o", label="Training")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel(f"{BEST_LOSS} loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["cer"], marker="o", label="Training")
    axes[1].plot(epochs, history["val_cer"], marker="o", label="Validation")
    axes[1].set_title(f"{title}: Character Error Rate")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("CER (lower is better)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_history(shallow_history, "Shallow MLP (2 hidden layers)")

In [ ]:
plot_history(deep_history, "Deep MLP (5 hidden layers)")

## 5. Discussion

*(Write after running the notebook, using the tables and plots above.)*

* **Best loss and optimizer:** which combination won the 3 x 3 grid, and by how much.
* **Learning rate:** what happened with the largest and smallest learning rates (too slow, unstable, or best).
* **Shallow vs deep:** which network has the lower test CER, and whether the extra layers helped or overfit
  (compare the gap between the training and validation curves).
* **Limitation:** an MLP only sees one 8-pixel frame at a time and has no context from neighbouring frames, so a high
  CER is expected. This motivates convolutional and recurrent models in the next milestones.